# Pawnee National Grassland Land Swap
## Pawnee Grassland Boundaries

- **Objective:**  
For this notebook, different shapefiles are queried to create different boundaries for federal and state parcels as well as a master and western Pawnee boundary shapefile to be used in all proceeding notebooks.  

- **Objective goals:**
    - Compile and process boundary datasets for the Pawnee National Grassland study area  
    - Create a unified **master boundary** for spatial analysis  
    - Generate subset boundaries (e.g., western Pawnee) for targeted analyses  
    - Prepare clean, reusable boundary layers for downstream workflows  

- **Author:** Max Warnock  
- **Code review and/or edits:** Kayleigh Ward  
- **Date:** April 9, 2026  
- **Last date of revision:** April 28, 2026  

---

### 🛠️ Prerequisites & Setup

**Mandatory Libraries:**
- `geopandas`
- `pandas`
- `shapely`
- `matplotlib`
- `requests` (for API data access)

**Environment:**
- Conda environment (e.g., `earth-analytics-python`) with geospatial dependencies installed  
- Internet connection required for querying parcel data  

**Data Sources:**
- USFS Administrative Boundaries (Pawnee National Grassland)  
- Weld County Parcels (ArcGIS REST Feature Service)  
- Supporting boundary shapefiles stored in `/data/boundaries/`  

**Related Notebooks:**
- Downstream notebooks (e.g., land values, species occurrences) depend on outputs from this notebook  

**Notes:**
- All layers are projected to a common CRS prior to processing  
- The master boundary serves as the **spatial constraint** for all analyses in this project  
- Boundary processing is designed to preserve spatial detail while enabling efficient clipping and overlay operations  

---

### 🏗️ Methodology

#### 1. Load and Inspect Boundary Data
- Import USFS boundary shapefiles and parcel data  
- Inspect geometry, CRS, and attribute structure  

#### 2. Query and Prepare Parcel Data
- Retrieve parcel data from Weld County API  
- Filter parcels by ownership (e.g., federal, state)  
- Clean and standardize attribute fields  

#### 3. Reproject and Align Spatial Data
- Convert all datasets to a common CRS  
- Ensure consistent spatial alignment across layers  

#### 4. Create Ownership-Based Boundaries
- Subset parcels by ownership type (federal, state)  
- Dissolve geometries where appropriate to create unified ownership boundaries  

#### 5. Construct Master Boundary
- Combine relevant boundary layers  
- Dissolve into a single **master boundary** polygon  
- Validate geometry and remove artifacts  

#### 6. Generate Subset Boundaries
- Clip master boundary to create the **western Pawnee boundary**  
- Export all finalized boundary layers for reuse  

---

### Reproducibility Notes
- All file paths are relative to the project root directory  
- Outputs are written to `/data/boundaries/boundary-data-final/`  
- This notebook should be run prior to any analysis notebooks that require spatial constraints  

---

### ⚡ Troubleshooting/Notes
- Ensure CRS consistency before performing spatial operations (common source of empty outputs) for this notebook  
- Large parcel datasets may take time to load and process  
- API queries may fail intermittently—rerun cells if needed  
- If geometries appear distorted, verify projection and transformation steps  

# Libraries

In [1]:
### file paths, OS operations, utilities
import os
import json
import pathlib
import zipfile
import time
from glob import glob
from getpass import getpass

### data handling 
import pandas as pd
import geopandas as gpd

### web requests / data download
import requests
import gdown

### geospatial visualization 
import holoviews as hv
import hvplot.pandas
import cartopy.crs as ccrs

### GBIF API access
import pygbif.occurrences as occ
import pygbif.species as species

# Primary Directory

In [2]:
### set up root file path
# Walk up from the current directory to find the repo root (contains .git)
_cwd = pathlib.Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / '.git').exists()),
    _cwd
)
os.chdir(repo_root)

data_dir = os.path.join(repo_root, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Repo root: {repo_root}')

Repo root: C:\Users\naho5798\Documents\Earth Data Cert\Final Project\Pawnee-Grasslands-Project


# Secondary Directories

In [7]:
### set a directory for the National Grassland boundary data
boundary_dir = os.path.join(data_dir, 'boundaries')
os.makedirs(boundary_dir, exist_ok=True)


### set an inner directory for boundary data processing
bound_process_dir = os.path.join(boundary_dir, 'boundary-data-processing')
os.makedirs(bound_process_dir, exist_ok=True)


### set a directory for the master boundary data
master_boundary_dir = os.path.join(bound_process_dir, 'master_boundary')
os.makedirs(master_boundary_dir, exist_ok=True)


### set a directory for the federal boundary data
federal_boundary_dir = os.path.join(bound_process_dir, 'federal_boundary')
os.makedirs(federal_boundary_dir, exist_ok=True)


### set a directory for the state boundary data
state_boundary_dir = os.path.join(bound_process_dir, 'state_boundary')
os.makedirs(state_boundary_dir, exist_ok=True)

### set a directory for the county parcel data
county_parcel_dir = os.path.join(bound_process_dir, 'county_parcels')
os.makedirs(county_parcel_dir, exist_ok=True)

### Master boundary data download

In [8]:
### Google Drive url
master_boundary_url = f"https://drive.google.com/uc?export=download&id=1gsL1tzZu6ZH28b3PQm4VgZzlY2GrkO8p"

### zip path
download_path = os.path.join(master_boundary_dir, "pawnee_master_boundary.zip")

### create a session
session = requests.Session()

### request the file
response = session.get(master_boundary_url, stream=True)
response.raise_for_status()

### save the zip file
with open(download_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print(f"Downloaded to: {download_path}")

### unzip the file
with zipfile.ZipFile(download_path, "r") as zip_ref:
    zip_ref.extractall(master_boundary_dir)

print(f"Extracted to: {master_boundary_dir}")

Downloaded to: C:\Users\naho5798\Documents\Earth Data Cert\Final Project\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\master_boundary\pawnee_master_boundary.zip
Extracted to: C:\Users\naho5798\Documents\Earth Data Cert\Final Project\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\master_boundary


In [9]:
### path to extracted folder
master_boundary_dir = os.path.join(master_boundary_dir, "pawnee_master_boundary")

### build full path
pawnee_master_boundary_path = os.path.join(master_boundary_dir, "pawnee_master_boundary.shp")

### read the shapefile
pawnee_master_boundary_gdf = gpd.read_file(pawnee_master_boundary_path)

### check it
pawnee_master_boundary_gdf.head()

,Id,Location,geometry
0,0,1.0,"POLYGON ((-11593123.97 5012571.209, -11542474...."
1,0,NaN,"POLYGON ((-11614609.509 4955044.708, -11616750..."


In [10]:
### make sure its projected in EPSG 4326
pawnee_master_boundary_gdf = pawnee_master_boundary_gdf.to_crs(epsg=4326)

### plot with hvplot
pawnee_master_boundary_gdf.hvplot(
    geo=True,
    tiles="EsriImagery",
    title="Pawnee National Grassland Boundaries",
    line_color="white",
    line_width=2,
    fill_alpha=0,
    width=600,
    height=500
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

### Download Weld County Parcel data

In [ ]:
# ============================================================
# PARCEL DATA SOURCE SELECTION
#
# Set parcel_source to choose how county parcel data is downloaded.
#
#   "weld_gis"     — Query from Weld Co. ArcGIS FeatureServer (default).
#                    Falls back to "google_drive" automatically if the 
#                    server is unreachable.
#
#   "google_drive" — Static backup on Google Drive (Max's copy).
#                    Included just in case the Weld Co. data goes down.
#
# Parcel URLs:
WELD_GIS_URL     = "https://services.arcgis.com/ewjSqmSyHJnkfBLL/arcgis/rest/services/Parcels_open_data/FeatureServer/0/query"
GOOGLE_DRIVE_URL = "https://drive.google.com/uc?export=download&id=1B1xTWb-Dfy9vJBFPB3hiSRex3j6eZ__2"
# ============================================================

parcel_source = "weld_gis"   # change this to google_drive if you want to use that version

print(f"Parcel source set to: '{parcel_source}'")

Parcel source set to: 'weld_gis'


In [ ]:
## Code from Kayleigh's notebook 04_land_value.ipynb transferred to here for handling
## ESRI download from Weld Co.
def polygon_to_esri_json(geom):
    """Convert a shapely Polygon/MultiPolygon to Esri JSON rings for use in ArcGIS queries."""
    polys = list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]
    rings = []
    for poly in polys:
        rings.append([[float(x), float(y)] for x, y in poly.exterior.coords])
        for interior in poly.interiors:
            rings.append([[float(x), float(y)] for x, y in interior.coords])
    return {"rings": rings, "spatialReference": {"wkid": 4326}}

## Pulls parcel data with a small buffer on the master boundary 
## to make sure we don't miss anything.
def query_parcels_by_boundary(boundary_gdf, buffer_deg=0.0, out_fields="*", batch_size=1000, max_pages=100):
    """
    Query the Weld Co. ArcGIS FeatureServer for all parcels that intersect
    the given boundary GeoDataFrame. Requests all available fields (out_fields='*').

    Parameters
    ----------
    boundary_gdf : GeoDataFrame
        The boundary to use as a spatial filter (e.g. pawnee_master_boundary_gdf).
        Must be in EPSG:4326.
    buffer_deg : float
        Buffer in decimal degrees set to 0.01 (~ 1 km). Get parcels just outside the Pawnee grassland.
    """
    frames = []
    offset = 0
    out_str = ",".join(out_fields) if isinstance(out_fields, list) else out_fields

    ### Dissolve boundary to a single geometry and buffer
    query_geom = boundary_gdf.to_crs(epsg=4326).geometry.union_all()
    if buffer_deg > 0:
        query_geom = query_geom.buffer(buffer_deg)
    esri_json = polygon_to_esri_json(query_geom)

    for page in range(max_pages):
        params = {
            "where": "1=1",
            "geometry": json.dumps(esri_json),
            "geometryType": "esriGeometryPolygon",
            "spatialRel": "esriSpatialRelIntersects",
            "outFields": out_str,
            "returnGeometry": "true",
            "outSR": 4326,
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": batch_size,
        }
        r = requests.post(WELD_GIS_URL, data=params, timeout=120)

        if not r.ok:
            print("HTTP status:", r.status_code)
            print("Response preview:", r.text[:300])
            r.raise_for_status()

        data = r.json()

        if "error" in data:
            raise RuntimeError(f"ArcGIS error: {data['error']}")

        features = data.get("features", [])
        if not features:
            break

        batch_gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
        frames.append(batch_gdf)
        total = sum(len(f) for f in frames)
        print(f"Page {page + 1}: fetched {len(features):,} parcels (total: {total:,})")

        if len(features) < batch_size:
            break

        offset += batch_size

    if not frames:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    return gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs="EPSG:4326")

## Add in lat/long columns to downloaded data + crs to WGS84
def harmonize_parcel_schema(gdf):
    """
    Normalize a parcel GeoDataFrame for consistent downstream use:
    - Reproject to EPSG:4326 (Google Drive .shp is in Web Mercator)
    - Ensure latitude/longitude attribute columns are present
      (derived from geometry centroids if absent from the source)
    All columns from the source are kept as-is — no filtering applied.
    """
    gdf = gdf.copy()

    ### Reproject to EPSG:4326
    if gdf.crs and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    ### Add latitude/longitude from geometry centroids if not present
    if "latitude" not in gdf.columns:
        centroids = gdf.geometry.centroid
        gdf["latitude"]  = centroids.y
        gdf["longitude"] = centroids.x

    return gdf


### Load from Google Drive backup
def _load_from_google_drive():
    zip_path = os.path.join(county_parcel_dir, "county_parcels.zip")
    gdown.download(GOOGLE_DRIVE_URL, zip_path, quiet=False)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(county_parcel_dir)
    shp_path = os.path.join(county_parcel_dir, "Parcels_open_data.shp")
    return gpd.read_file(shp_path)


### Download
county_parcel_gdf = None
source_used = None

if parcel_source == "weld_gis":
    try:
        ### Pull parcels that only intersect the master boundary + buffer.
        county_parcel_gdf = query_parcels_by_boundary(
            pawnee_master_boundary_gdf,
            buffer_deg=0.01
        )
        source_used = "Weld Co. ArcGIS FeatureServer (Pawnee boundary)"
        print(f"Downloaded {len(county_parcel_gdf):,} parcels from {source_used}")
    except Exception as e:
        print(f"WARNING: Weld Co. ArcGIS download failed: {e}")
        print("Falling back to Google Drive backup...")
        county_parcel_gdf = _load_from_google_drive()
        source_used = "Google Drive backup (auto-fallback)"
        print(f"Loaded {len(county_parcel_gdf):,} parcels from {source_used}")

elif parcel_source == "google_drive":
    county_parcel_gdf = _load_from_google_drive()
    source_used = "Google Drive"
    print(f"Loaded {len(county_parcel_gdf):,} parcels from {source_used}")

else:
    raise ValueError(f"Unknown parcel_source: '{parcel_source}'. Use 'weld_gis' or 'google_drive'.")

### Apply lat/long + reprojection
county_parcel_gdf = harmonize_parcel_schema(county_parcel_gdf)
print(f"Schema standardized — {len(county_parcel_gdf):,} parcels, {len(county_parcel_gdf.columns)} columns")

Page 1: fetched 1,000 parcels (total: 1,000)
Page 2: fetched 1,000 parcels (total: 2,000)
Page 3: fetched 1,000 parcels (total: 3,000)
Page 4: fetched 1,000 parcels (total: 4,000)
Page 5: fetched 167 parcels (total: 4,167)
Downloaded 4,167 parcels from Weld Co. ArcGIS FeatureServer (Pawnee boundary)
Schema standardized — 4,167 parcels, 62 columns


In [ ]:
### Save the raw parcel data as a .gpkg
raw_parcel_path = os.path.join(county_parcel_dir, "county_parcels_raw.gpkg")
county_parcel_gdf.to_file(raw_parcel_path, driver="GPKG")
print(f"Saved raw parcel data ({source_used}) to: {raw_parcel_path}")

### Check it
county_parcel_gdf.head()

INFO:Created 4,167 records


Saved raw parcel data (Weld Co. ArcGIS FeatureServer (Pawnee boundary)) to: C:\Users\naho5798\Documents\Earth Data Cert\Final Project\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\county_parcels\county_parcels_raw.gpkg


,geometry,OBJECTID,PARCEL,MHSPACE,ACCOUNTTYP,ACCOUNTNO,NAME,ADDRESS1,ADDRESS2,CITY,...,RECEPTION_,AddressPre,LGLANDASD,LGIMPASD,TOTALLGASD,SCLANDASD,SCIMPASD,TOTALSCASD,Shape__Area,Shape__Length
0,"POLYGON ((-103.68329 41.0017, -103.67373 41.00...",1,002919000001,,R,R0000186,GREEN ELSIE KLINGINSMITH (1/3 INT),530 MCKINLEY ST,,STERLING,...,,,1790,0,1790,1790,0,1790,2.960275e+06,7560.798175
1,"POLYGON ((-103.68547 40.99295, -103.68546 40.9...",2,002919000002,,R,R0000286,U S A,2850 YOUNGFIELD ST,,LAKEWOOD,...,,,10030,0,10030,10030,0,10030,1.777779e+04,533.312931
2,"POLYGON ((-103.64981 40.99951, -103.64986 40.9...",3,002920000003,,R,R0000386,GREEN ELSIE KLINGINSMITH (1/3 INT),530 MCKINLEY ST,,STERLING,...,,,1820,0,1820,1820,0,1820,2.929635e+06,7008.613912
3,"POLYGON ((-103.63083 40.99958, -103.63081 40.9...",4,002921000001,,R,R0000586,GREEN ELSIE KLINGINSMITH (1/3 INT),530 MCKINLEY ST,,STERLING,...,,,1980,0,1980,1980,0,1980,2.915064e+06,6986.967992
4,"POLYGON ((-103.61175 40.99975, -103.61174 40.9...",5,002922000002,,R,R0000686,SHEFFLER RANCH LLC,65295 COUNTY ROAD 135,,NEW RAYMER,...,,,1790,8090,9880,1790,8910,10700,2.878536e+06,6962.167505


### Select federal parcels

In [14]:
### select only federal owned parcels
pawnee_fed = county_parcel_gdf[county_parcel_gdf["NAME"] == "U S A"].copy()

### check the result
pawnee_fed

,geometry,OBJECTID,PARCEL,MHSPACE,ACCOUNTTYP,ACCOUNTNO,NAME,ADDRESS1,ADDRESS2,CITY,...,RECEPTION_,AddressPre,LGLANDASD,LGIMPASD,TOTALLGASD,SCLANDASD,SCIMPASD,TOTALSCASD,Shape__Area,Shape__Length
1,"POLYGON ((-103.68547 40.99295, -103.68546 40.9...",2,002919000002,,R,R0000286,U S A,2850 YOUNGFIELD ST,,LAKEWOOD,...,,,10030,0,10030,10030,0,10030,1.777779e+04,533.312931
6,"POLYGON ((-103.59331 41.00007, -103.59359 41.0...",7,002923000008,,R,R0001186,U S A,2850 YOUNGFIELD ST,,LAKEWOOD,...,,,10030,0,10030,10030,0,10030,1.778019e+04,533.401611
27,"POLYGON ((-103.78158 40.99978, -103.78158 40.9...",28,003120000002,,R,R0004786,U S A,2850 YOUNGFIELD ST,,LAKEWOOD,...,,,10030,0,10030,10030,0,10030,1.778136e+04,533.367067
33,"POLYGON ((-103.7402 40.99871, -103.74025 40.99...",34,003122000002,,R,R0005486,U S A,2850 YOUNGFIELD ST,,LAKEWOOD,...,,,38000,0,38000,38000,0,38000,8.564874e+05,4274.862742
42,"POLYGON ((-103.70694 40.98089, -103.70697 40.9...",43,003126000002,,R,R0006586,U S A,2850 YOUNGFIELD ST,,LAKEWOOD,...,,,57320,0,57320,57320,0,57320,2.270989e+06,6388.881213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3676,"MULTIPOLYGON (((-104.62011 40.68914, -104.6217...",6606,055102000011,,R,R0607286,U S A,2850 YOUNGFIELD ST,,LAKEWOOD,...,,,8130,0,8130,8130,0,8130,4.313466e+05,3753.182333
3702,"POLYGON ((-104.60573 40.68176, -104.60093 40.6...",6681,055112000006,,R,R0610886,U S A,2850 YOUNGFIELD ST,,LAKEWOOD,...,,,61620,0,61620,61620,0,61620,3.389359e+06,8504.302139
3704,"POLYGON ((-104.61525 40.66745, -104.61046 40.6...",6683,055113000008,,R,R0611086,U S A,2850 YOUNGFIELD ST,,LAKEWOOD,...,,,30940,0,30940,30940,0,30940,1.704883e+06,6399.347654
3707,"POLYGON ((-104.62005 40.66387, -104.62005 40.6...",6686,055114000009,,R,R0611486,U S A,2850 YOUNGFIELD ST,,LAKEWOOD,...,,,41600,0,41600,41600,0,41600,2.284068e+06,6414.545435


In [15]:
### make sure its projected in EPSG 4326
pawnee_fed = pawnee_fed.to_crs(epsg=4326)

### clip to the master boundary
pawnee_fed = gpd.clip(
    pawnee_fed,
    pawnee_master_boundary_gdf
)

### plot with hvplot
pawnee_fed.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Pawnee National Grassland Federally Owned Boundaries',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

### Select state owned parcels

In [16]:
### select only state owned parcels
pawnee_state = county_parcel_gdf[county_parcel_gdf["NAME"] == "COLORADO STATE OF"].copy()

### check the result
pawnee_state

,geometry,OBJECTID,PARCEL,MHSPACE,ACCOUNTTYP,ACCOUNTNO,NAME,ADDRESS1,ADDRESS2,CITY,...,RECEPTION_,AddressPre,LGLANDASD,LGIMPASD,TOTALLGASD,SCLANDASD,SCIMPASD,TOTALSCASD,Shape__Area,Shape__Length
14,"POLYGON ((-103.63082 40.98872, -103.63084 40.9...",15,002928000006,,R,R0002586,COLORADO STATE OF,1127 N SHERMAN ST STE 300,,DENVER,...,,,57320,0,57320,57320,0,57320,2.270155e+06,6396.935020
22,"POLYGON ((-103.61182 40.97438, -103.61184 40.9...",23,002934000004,,R,R0004386,COLORADO STATE OF,1127 N SHERMAN ST STE 300,,DENVER,...,,,41310,0,41310,41310,0,41310,2.265107e+06,6393.664036
24,"POLYGON ((-103.57352 40.97447, -103.57365 40.9...",25,002936000004,,R,R0004586,COLORADO STATE OF,1127 N SHERMAN ST STE 300,,DENVER,...,,,76640,0,76640,76640,0,76640,4.535905e+06,8520.295240
60,"POLYGON ((-103.68771 40.97383, -103.68763 40.9...",61,003136000004,,R,R0009086,COLORADO STATE OF,1127 N SHERMAN ST STE 300,,DENVER,...,,,2650,0,2650,2650,0,2650,4.580195e+06,8560.641430
134,"POLYGON ((-103.91615 40.97221, -103.91606 40.9...",135,003536000005,,R,R0018186,COLORADO STATE OF,1127 N SHERMAN ST STE 300,,DENVER,...,,,81640,0,81640,81640,0,81640,4.413393e+06,8405.565205
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3493,"POLYGON ((-104.52457 40.67859, -104.5246 40.67...",6423,054910000008,,R,R0594486,COLORADO STATE OF,1127 N SHERMAN ST STE 300,,DENVER,...,,,15340,0,15340,15340,0,15340,8.447188e+05,4245.379458
3500,"POLYGON ((-104.50546 40.66424, -104.50557 40.6...",6430,054914000002,,R,R0595386,COLORADO STATE OF,1127 N SHERMAN ST STE 300,,DENVER,...,,,57070,0,57070,57070,0,57070,3.110654e+06,9580.787157
3612,"POLYGON ((-104.50163 40.63883, -104.50651 40.6...",6542,054924000005,,R,R0603486,COLORADO STATE OF,1127 N SHERMAN ST STE 300,,DENVER,...,,,30680,0,30680,30680,0,30680,1.719880e+06,5350.333851
3669,"POLYGON ((-104.4874 40.62062, -104.48762 40.61...",6599,054936000003,,R,R0606386,COLORADO STATE OF,1127 N SHERMAN ST STE 300,,DENVER,...,,,77610,0,77610,77610,0,77610,4.231522e+06,9547.875255


In [17]:
### make sure its projected in EPSG 4326
pawnee_state = pawnee_state.to_crs(epsg=4326)

### clip to the master boundary
pawnee_state = gpd.clip(
    pawnee_state,
    pawnee_master_boundary_gdf
)

### plot with hvplot
pawnee_state.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Pawnee National Grassland State Owned Boundaries',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [18]:
### make sure its projected in EPSG 4326
county_parcel_gdf = county_parcel_gdf.to_crs(epsg=4326)

### clip to the master boundary
county_parcel_gdf = gpd.clip(
    county_parcel_gdf,
    pawnee_master_boundary_gdf
)

### plot all parcels
county_parcel_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Pawnee National Grassland Parcel Boundaries',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [19]:
### state plot
state_plot = pawnee_state.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="blue",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="State"
)

### federal plot
fed_plot = pawnee_fed.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="green",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="Federal"
)

### master boundary plot
master_plot = pawnee_master_boundary_gdf.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    fill_alpha=0,
    line_color="white",
    line_width=2,
    label="Master Boundary"
)

### create the plot
pawnee_boundary_plot = (hv.element.tiles.EsriImagery() * state_plot * fed_plot * master_plot
).opts(
    title="Pawnee State, Federal, and Master Boundaries",
    width=700,
    height=550,
    legend_position="top_left"
)

### save an interactive html version of the plot

figures_dir = os.path.join(repo_root, 'figures')

boundary_fig_dir = os.path.join(figures_dir, 'boundary_figures')
os.makedirs(boundary_fig_dir, exist_ok=True)

boundary_plot_path = os.path.join(boundary_fig_dir, 'pawnee_boundary_plot.html')
hv.save(pawnee_boundary_plot, boundary_plot_path)

print(f'Saved clipped GBIF map to: {boundary_plot_path}')

pawnee_boundary_plot

Saved clipped GBIF map to: C:\Users\naho5798\Documents\Earth Data Cert\Final Project\Pawnee-Grasslands-Project\figures\boundary_figures\pawnee_boundary_plot.html


:Overlay
   .Tiles.I                  :Tiles   [x,y]
   .Polygons.State           :Polygons   [Longitude,Latitude]
   .Polygons.Federal         :Polygons   [Longitude,Latitude]
   .Polygons.Master_Boundary :Polygons   [Longitude,Latitude]

### Create file structure for Pawnee processed data

In [20]:
### set a directory for final boundary data
bound_final_dir = os.path.join(boundary_dir, 'boundary-data-final')
os.makedirs(bound_final_dir, exist_ok=True)


### set a directory for the master boundary data
master_bound_final_dir = os.path.join(bound_final_dir, 'master_boundary')
os.makedirs(master_bound_final_dir, exist_ok=True)


### set a directory for the federal boundary data
fed_bound_final_dir = os.path.join(bound_final_dir, 'federal_boundary')
os.makedirs(fed_bound_final_dir, exist_ok=True)


### set a directory for the state boundary data
state_bound_final_dir = os.path.join(bound_final_dir, 'state_boundary')
os.makedirs(state_bound_final_dir, exist_ok=True)


### set a directory for the parcel boundary data
parcel_bound_final_dir = os.path.join(bound_final_dir, 'parcel_boundary')
os.makedirs(parcel_bound_final_dir, exist_ok=True)

### Save processed data in these folders

In [22]:
### save master boundary
pawnee_master_path = os.path.join(master_bound_final_dir, "pawnee_master.shp")
pawnee_master_boundary_gdf.to_file(pawnee_master_path)

### save fed boundary
pawnee_fed_path = os.path.join(fed_bound_final_dir, "pawnee_fed.shp")
pawnee_fed.to_file(pawnee_fed_path)

### save state boundary
pawnee_state_path = os.path.join(state_bound_final_dir, "pawnee_state.shp")
pawnee_state.to_file(pawnee_state_path)

### save state boundary
pawnee_parcel_path = os.path.join(parcel_bound_final_dir, "pawnee_parcel.shp")
county_parcel_gdf.to_file(pawnee_parcel_path)

INFO:Created 2 records
C:\Users\naho5798\AppData\Local\Temp\ipykernel_250444\1662622620.py:7: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  pawnee_fed.to_file(pawnee_fed_path)
c:\Users\naho5798\AppData\Local\miniconda3\envs\pawnee-grasslands\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Shape__Area' to 'Shape__Are'
  ogr_write(
c:\Users\naho5798\AppData\Local\miniconda3\envs\pawnee-grasslands\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Shape__Length' to 'Shape__Len'
  ogr_write(
INFO:Created 598 records
C:\Users\naho5798\AppData\Local\Temp\ipykernel_250444\1662622620.py:11: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  pawnee_state.to_file(pawnee_state_path)
c:\Users\naho5798\AppData\Local\miniconda3\envs\pawnee-grasslands\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laun

### Universal Calls for Pawnee Boundary Data

In [23]:
### root dir
data_dir = os.path.join(repo_root, 'data')

### main boundary dir
boundary_dir = os.path.join(data_dir, 'boundaries')

### boundary dir for processed data for Pawnee
boundary_dir_final = os.path.join(boundary_dir, 'boundary-data-final')


### MASTER
### master boundary dir
master_bound = os.path.join(boundary_dir_final, 'master_boundary')

### master boundary shapefile
master_bound_path = os.path.join(master_bound, "pawnee_master.shp")

### master boundary convert to gdf
master_bound_gdf = gpd.read_file(master_bound_path)


### FEDERAL
### federal boundary dir
federal_bound = os.path.join(boundary_dir_final, 'federal_boundary')

### federal boundary shapefile 
federal_bound_path = os.path.join(federal_bound, 'pawnee_fed.shp')

### federal boundary convert to gdf
federal_bound_gdf = gpd.read_file(federal_bound_path)


### STATE
### state boundary dir
state_bound = os.path.join(boundary_dir_final, 'state_boundary')

### state boundary shapefile 
state_bound_path = os.path.join(state_bound, 'pawnee_state.shp')

### state boundary convert to gdf
state_bound_gdf = gpd.read_file(state_bound_path)


### PARCEL
### parcel boundary dir
parcel_bound = os.path.join(boundary_dir_final, 'parcel_boundary')

### parcel boundary shapefile 
parcel_bound_path = os.path.join(parcel_bound, 'pawnee_parcel.shp')

### parcel boundary convert to gdf
parcel_bound_gdf = gpd.read_file(parcel_bound_path)

### Select only the Western Pawnee area

In [24]:
### select only western pawnee area
pawnee_west = pawnee_master_boundary_gdf[pawnee_master_boundary_gdf["Location"].isna()].copy()
pawnee_west

,Id,Location,geometry
1,0,NaN,"POLYGON ((-104.33581 40.6104, -104.35504 40.61..."


In [25]:
### clip state to the western pawnee
pawnee_state_west = gpd.clip(
    pawnee_state,
    pawnee_west
)

In [26]:
### clip fed to the western pawnee
pawnee_fed_west = gpd.clip(
    pawnee_fed,
    pawnee_west
)

In [27]:
### clip parcels to the western pawnee
pawnee_parcel_west = gpd.clip(
    county_parcel_gdf,
    pawnee_west
)

In [28]:
### plot all three again for the western pawnee

### state plot
state_plot_west = pawnee_state_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="blue",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="State"
)

### federal plot
fed_plot_west = pawnee_fed_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="green",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="Federal"
)

### master boundary plot west
master_plot_west = pawnee_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    fill_alpha=0,
    line_color="white",
    line_width=2,
    label="Master Boundary"
)

parcel_plot_west = pawnee_parcel_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    fill_alpha=0,
    line_color='white',
    line_width=1,
    label="Parcels"
)

### create the plot
west_pawnee_plot = (hv.element.tiles.EsriImagery() * parcel_plot_west * state_plot_west * fed_plot_west * master_plot_west
).opts(
    title="West Pawnee State, Federal, and Master Boundaries",
    width=700,
    height=550,
    legend_position="top_left"
)

### save an interactive html version of the plot

west_boundary_plot_path = os.path.join(boundary_fig_dir, 'west_pawnee_boundary_plot.html')
hv.save(west_pawnee_plot, west_boundary_plot_path)

print(f'Saved clipped GBIF map to: {west_boundary_plot_path}')

west_pawnee_plot

Saved clipped GBIF map to: C:\Users\naho5798\Documents\Earth Data Cert\Final Project\Pawnee-Grasslands-Project\figures\boundary_figures\west_pawnee_boundary_plot.html


:Overlay
   .Tiles.I                  :Tiles   [x,y]
   .Polygons.Parcels         :Polygons   [Longitude,Latitude]
   .Polygons.State           :Polygons   [Longitude,Latitude]
   .Polygons.Federal         :Polygons   [Longitude,Latitude]
   .Polygons.Master_Boundary :Polygons   [Longitude,Latitude]

### Create file structure for Pawnee West processed data

In [29]:
### set a directory for final boundary west data 
bound_final_dir_west = os.path.join(boundary_dir, 'boundary-data-final-west')
os.makedirs(bound_final_dir_west, exist_ok=True)


### set a directory for the master boundary west data
master_bound_final_dir_west = os.path.join(bound_final_dir_west, 'master_boundary_west')
os.makedirs(master_bound_final_dir_west, exist_ok=True)


### set a directory for the federal boundary data
fed_bound_final_dir_west = os.path.join(bound_final_dir_west, 'federal_boundary_west')
os.makedirs(fed_bound_final_dir_west, exist_ok=True)


### set a directory for the state boundary data
state_bound_final_dir_west = os.path.join(bound_final_dir_west, 'state_boundary_west')
os.makedirs(state_bound_final_dir_west, exist_ok=True)


### set a directory for the parcel boundary data
parcel_bound_final_dir_west = os.path.join(bound_final_dir_west, 'parcel_boundary_west')
os.makedirs(parcel_bound_final_dir_west, exist_ok=True)

### Save Pawnee West processed data in these folders

In [30]:
### save master boundary
pawnee_west_path = os.path.join(master_bound_final_dir, "pawnee_master_west.shp")
pawnee_west.to_file(pawnee_west_path)

### save fed boundary
pawnee_fed_west_path = os.path.join(fed_bound_final_dir, "pawnee_fed_west.shp")
pawnee_fed_west.to_file(pawnee_fed_west_path)

### save state boundary
pawnee_state_west_path = os.path.join(state_bound_final_dir, "pawnee_state_west.shp")
pawnee_state_west.to_file(pawnee_state_west_path)

### save state boundary
pawnee_parcel_west_path = os.path.join(parcel_bound_final_dir, "pawnee_parcel_west.shp")
pawnee_parcel_west.to_file(pawnee_parcel_west_path)

INFO:Created 1 records
C:\Users\naho5798\AppData\Local\Temp\ipykernel_250444\1776546567.py:7: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  pawnee_fed_west.to_file(pawnee_fed_west_path)
c:\Users\naho5798\AppData\Local\miniconda3\envs\pawnee-grasslands\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Shape__Area' to 'Shape__Are'
  ogr_write(
c:\Users\naho5798\AppData\Local\miniconda3\envs\pawnee-grasslands\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Shape__Length' to 'Shape__Len'
  ogr_write(
INFO:Created 246 records
C:\Users\naho5798\AppData\Local\Temp\ipykernel_250444\1776546567.py:11: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  pawnee_state_west.to_file(pawnee_state_west_path)
c:\Users\naho5798\AppData\Local\miniconda3\envs\pawnee-grasslands\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarn

### Universal Calls for Pawnee West Boundary Data

In [31]:
### root dir
data_dir = os.path.join(repo_root, 'data')

### main boundary dir
boundary_dir = os.path.join(data_dir, 'boundaries')

### boundary dir for processed data for Western Pawnee
boundary_dir_west = os.path.join(boundary_dir, 'boundary-data-final-west')


### MASTER
### master boundary dir
master_bound_west = os.path.join(boundary_dir_west, 'master_boundary')

### master boundary shapefile
master_bound_west_path = os.path.join(master_bound_west, "pawnee_master_west.shp")

### master boundary convert to gdf
master_bound_west_gdf = gpd.read_file(master_bound_west_path)


### FEDERAL
### federal boundary dir
federal_bound_west = os.path.join(boundary_dir_west, 'federal_boundary')

### federal boundary shapefile 
federal_bound_west_path = os.path.join(federal_bound_west, 'pawnee_fed_west.shp')

### federal boundary convert to gdf
federal_bound_west_gdf = gpd.read_file(federal_bound_west_path)


### STATE
### state boundary dir
state_bound_west = os.path.join(boundary_dir_west, 'state_boundary')

### state boundary shapefile 
state_bound_west_path = os.path.join(state_bound_west, 'pawnee_state_west.shp')

### state boundary convert to gdf
state_bound_west_gdf = gpd.read_file(state_bound_west_path)


### PARCEL
### parcel boundary dir
parcel_bound_west = os.path.join(boundary_dir_west, 'parcel_boundary')

### parcel boundary shapefile 
parcel_bound_west_path = os.path.join(parcel_bound_west, 'pawnee_parcel_west.shp')

### parcel boundary convert to gdf
parcel_bound_west_gdf = gpd.read_file(parcel_bound_west_path)